# 🔧 Module 2.1: Filters and Convolutions — The Mathematical Heart of Image Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_02_Image_Processing_Basics/01_Filters_And_Convolutions/filters_and_convolutions.ipynb)

---

## 🎯 Learning Objectives
1. Understand convolution from first principles (full math derivation)
2. Implement 2D convolution from scratch
3. Common filters: blur, sharpen, emboss, edge detection
4. Filters as RL actions — an agent choosing which kernel to apply

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless -q

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# CIFAR-10: 60,000 real photographs in 10 classes (auto-downloads ~170MB)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10 loaded: {len(cifar10)} real photos (32x32x3)")
print(f"   Classes: {cifar10.classes}")

# scikit-image real test images
from skimage import data
sample_images = {
    'astronaut': data.astronaut(),
    'camera': data.camera(), 
    'coins': data.coins(),
    'horse': data.horse(),
}
print(f"✅ scikit-image: {len(sample_images)} real test images loaded")

## Deep Mathematical Derivation: Convolution from First Principles

### Step 1: 1D Continuous Convolution (Definition)
$$(f * g)(t) = \int_{-\infty}^{\infty} f(\tau) \cdot g(t - \tau) \, d\tau$$

**Intuition:** Slide $g$ across $f$, multiply pointwise, integrate. The "flip-and-drag" operation.

### Step 2: 2D Discrete Convolution for Images
For image $I$ and kernel $K$ of size $(2a+1) \times (2b+1)$:
$$(I * K)[x, y] = \sum_{i=-a}^{a} \sum_{j=-b}^{b} I[x-i, y-j] \cdot K[i, j]$$

**Note:** Convolution flips the kernel. Correlation does NOT:
$$(I \star K)[x, y] = \sum_{i=-a}^{a} \sum_{j=-b}^{b} I[x+i, y+j] \cdot K[i, j]$$

For symmetric kernels (Gaussian, Laplacian), convolution = correlation.

### Step 3: Convolution Theorem (Full Proof)
**Theorem:** Convolution in the spatial domain equals multiplication in the frequency domain:
$$\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$$

**Proof:**
$$\mathcal{F}\{f * g\}(\omega) = \int_{-\infty}^{\infty} \left[\int_{-\infty}^{\infty} f(\tau) g(t-\tau) d\tau \right] e^{-j\omega t} dt$$

$$= \int_{-\infty}^{\infty} f(\tau) \left[\int_{-\infty}^{\infty} g(t-\tau) e^{-j\omega t} dt\right] d\tau$$

Substituting $u = t - \tau$, so $dt = du$:
$$= \int_{-\infty}^{\infty} f(\tau) e^{-j\omega\tau} d\tau \cdot \int_{-\infty}^{\infty} g(u) e^{-j\omega u} du = F(\omega) \cdot G(\omega) \quad \blacksquare$$

**Computational implication:** Naive 2D convolution is $O(N^2 K^2)$. Via FFT:
$$O(N^2 \log N) \ll O(N^2 K^2) \text{ for large } K$$

### Step 4: Gaussian Kernel Derivation
The 2D Gaussian kernel:
$$G(x, y; \sigma) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{x^2 + y^2}{2\sigma^2}\right)$$

**Proof of separability:** $G(x,y;\sigma) = G(x;\sigma) \cdot G(y;\sigma)$

This means 2D Gaussian convolution can be decomposed into two 1D convolutions:
$$I * G_{2D} = (I * G_x) * G_y$$

**Proof of computational savings:**
- 2D: $O(N^2 K^2)$ operations
- Separable: $O(N^2 \cdot 2K) = O(N^2 K)$ — linear in kernel size! $\blacksquare$

### Step 5: Cascaded Gaussian Property
**Theorem:** Convolving with $G(\sigma_1)$ then $G(\sigma_2)$ equals convolving once with $G(\sigma)$:
$$G(\sigma_1) * G(\sigma_2) = G\left(\sqrt{\sigma_1^2 + \sigma_2^2}\right)$$

**Proof (via Fourier transform):**
$$\mathcal{F}\{G(\sigma)\}(\omega) = \exp\left(-\frac{\sigma^2 \omega^2}{2}\right)$$

$$\mathcal{F}\{G(\sigma_1) * G(\sigma_2)\} = \exp\left(-\frac{\sigma_1^2\omega^2}{2}\right) \cdot \exp\left(-\frac{\sigma_2^2\omega^2}{2}\right) = \exp\left(-\frac{(\sigma_1^2+\sigma_2^2)\omega^2}{2}\right) \quad \blacksquare$$

### Step 6: Sharpening Kernel Derivation
**Unsharp masking:** $I_{\text{sharp}} = I + \alpha(I - I * G) = (1+\alpha)I - \alpha(I * G)$

The Laplacian sharpening kernel is derived from the second derivative:
$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

Discretizing: $\frac{\partial^2 I}{\partial x^2} \approx I[x+1,y] - 2I[x,y] + I[x-1,y]$

$$\nabla^2 \approx \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

**Sharpening:** $I_{\text{sharp}} = I - \alpha \nabla^2 I$, with kernel:
$$K_{\text{sharp}} = \begin{bmatrix} 0 & -\alpha & 0 \\ -\alpha & 1+4\alpha & -\alpha \\ 0 & -\alpha & 0 \end{bmatrix} \quad \blacksquare$$

### RL Connection: Kernels as Actions
An RL agent can choose from a discrete action space of kernels:
$$\mathcal{A} = \{K_{\text{blur}}, K_{\text{sharpen}}, K_{\text{edge}}, K_{\text{emboss}}, K_{\text{identity}}\}$$

The **transition function** is deterministic: $s_{t+1} = s_t * a_t$ (convolve current image state with chosen kernel action).

## Convolution Theorem — Complete Proof and Computational Implications

The convolution theorem is arguably the most important result in signal processing. It reveals that convolution — the fundamental operation of image filtering — is simply pointwise multiplication in the frequency domain.

### Step 1: The 2D Discrete Fourier Transform (DFT)

For an image $I[m, n]$ of size $M \times N$:

$$F[u, v] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} I[m, n] \cdot e^{-j2\pi\left(\frac{um}{M} + \frac{vn}{N}\right)}$$

**Inverse DFT:**
$$I[m, n] = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F[u, v] \cdot e^{j2\pi\left(\frac{um}{M} + \frac{vn}{N}\right)}$$

### Step 2: Proof of the Convolution Theorem

**Theorem:** $\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$

**Full proof (continuous case, then discretized):**

$$\mathcal{F}\{f * g\}(\omega) = \int_{-\infty}^{\infty} \left[\int_{-\infty}^{\infty} f(\tau) g(t - \tau) \, d\tau\right] e^{-j\omega t} \, dt$$

Exchange order of integration (justified by Fubini's theorem when $f, g \in L^1$):

$$= \int_{-\infty}^{\infty} f(\tau) \left[\int_{-\infty}^{\infty} g(t - \tau) e^{-j\omega t} \, dt\right] d\tau$$

Substitute $u = t - \tau$, so $t = u + \tau$, $dt = du$:

$$= \int_{-\infty}^{\infty} f(\tau) \left[\int_{-\infty}^{\infty} g(u) e^{-j\omega(u + \tau)} \, du\right] d\tau$$

$$= \int_{-\infty}^{\infty} f(\tau) e^{-j\omega\tau} \, d\tau \cdot \int_{-\infty}^{\infty} g(u) e^{-j\omega u} \, du = F(\omega) \cdot G(\omega) \quad \blacksquare$$

### Step 3: Computational Complexity Analysis

**Direct spatial-domain convolution** of an $N \times N$ image with a $K \times K$ kernel:
$$T_{\text{spatial}} = O(N^2 K^2)$$

**Frequency-domain convolution via FFT:**
1. FFT of image: $O(N^2 \log N)$
2. FFT of kernel: $O(N^2 \log N)$ (or precomputed)
3. Pointwise multiplication: $O(N^2)$
4. Inverse FFT: $O(N^2 \log N)$

$$T_{\text{FFT}} = O(N^2 \log N)$$

**Crossover point:** FFT is faster when $K^2 > c \log N$ for some constant $c$. For a $1024 \times 1024$ image, this occurs around $K \approx 5$. For large kernels ($K > 10$), FFT-based convolution is dramatically faster.

### Step 4: The Dual Theorem

There is also a dual result: **multiplication in spatial domain equals convolution in frequency domain:**

$$\mathcal{F}\{f \cdot g\} = \frac{1}{2\pi} F(\omega) * G(\omega)$$

This explains why windowing (multiplying by a window function) causes spectral leakage (convolution smears the spectrum).

## Gaussian Blur as Solution to the Heat (Diffusion) Equation

The deep connection between Gaussian filtering and physics reveals why Gaussian blur is the natural smoothing operation — it is the unique solution to the heat equation on images.

### Step 1: The Isotropic Diffusion Equation

The heat equation in 2D:
$$\frac{\partial I}{\partial t} = D \nabla^2 I = D \left(\frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}\right)$$

where $I(x, y, t)$ is the image intensity at position $(x,y)$ and "time" $t$, and $D$ is the diffusion coefficient.

**Physical interpretation:** Intensity "flows" from high-intensity regions to low-intensity regions, like heat spreading through a metal plate. Over time, the image becomes smoother.

### Step 2: Solving via Fourier Transform

Taking the Fourier transform of both sides (in $x, y$):
$$\frac{\partial \hat{I}}{\partial t} = -D(u^2 + v^2)\hat{I}$$

This is an ODE in $t$ for each frequency $(u, v)$:
$$\hat{I}(u, v, t) = \hat{I}(u, v, 0) \cdot e^{-D(u^2 + v^2)t}$$

### Step 3: Inverse Fourier Transform Yields the Gaussian

The factor $e^{-D(u^2+v^2)t}$ is the Fourier transform of a Gaussian with $\sigma^2 = 2Dt$:

$$\mathcal{F}^{-1}\{e^{-D(u^2+v^2)t}\} = \frac{1}{4\pi Dt}\exp\left(-\frac{x^2+y^2}{4Dt}\right) = G(x, y; \sigma=\sqrt{2Dt})$$

Therefore:
$$I(x, y, t) = I(x, y, 0) * G(x, y; \sqrt{2Dt}) \quad \blacksquare$$

**The solution to the heat equation at time $t$ IS convolution with a Gaussian of width $\sigma = \sqrt{2Dt}$!**

### Step 4: Scale-Space Properties (Consequences)

This diffusion interpretation gives Gaussian blurring unique mathematical properties:

**1. Semigroup property:** $G(\sigma_1) * G(\sigma_2) = G(\sqrt{\sigma_1^2 + \sigma_2^2})$

*Proof:* Running diffusion for time $t_1$ then $t_2$ equals running for $t_1 + t_2$. Since $\sigma^2 = 2Dt$: $\sigma^2 = 2D(t_1 + t_2) = \sigma_1^2 + \sigma_2^2$. $\blacksquare$

**2. No new features:** Gaussian blur never creates new local extrema — it can only remove them. (Proved by the maximum principle for the heat equation.)

**3. Causality:** This is the defining property of **scale space**: information at coarser scales was always present at finer scales, never fabricated.

### Step 5: Why Gaussian Is Unique

**Theorem (Iijima, 1962; Lindeberg, 1994):** The Gaussian is the ONLY kernel that satisfies:
1. Linearity and shift invariance
2. Semigroup property (cascading)
3. Non-creation of local extrema
4. Rotational symmetry

Any other smoothing kernel violates at least one of these properties. This is why Gaussian blur is universally preferred for pre-processing in both traditional CV and RL-based vision.

## 1. Convolution — Definition

### Continuous 2D Convolution:
$$(f * g)(x, y) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(\tau_1, \tau_2) \cdot g(x - \tau_1, y - \tau_2) \, d\tau_1 \, d\tau_2$$

### Discrete 2D Convolution:
$$(I * K)[m, n] = \sum_{i=-k}^{k} \sum_{j=-k}^{k} I[m-i, n-j] \cdot K[i, j]$$

Where:
- $I$ is the input image
- $K$ is the kernel (filter) of size $(2k+1) \times (2k+1)$
- The output at $(m,n)$ is the weighted sum of pixel neighborhood

### Cross-Correlation (what libraries actually compute):
$$(I \star K)[m, n] = \sum_{i=-k}^{k} \sum_{j=-k}^{k} I[m+i, n+j] \cdot K[i, j]$$

Note: Convolution = Cross-correlation with flipped kernel.

### Properties:
1. **Commutative:** $f * g = g * f$
2. **Associative:** $(f * g) * h = f * (g * h)$
3. **Distributive:** $f * (g + h) = f*g + f*h$
4. **Shift-invariant:** Same operation everywhere

## Sharpening Kernel Derivation — From Unsharp Mask to Laplacian Enhancement

Image sharpening makes edges and fine details more prominent. Here we derive sharpening kernels rigorously from the unsharp masking principle.

### Step 1: Unsharp Masking Principle

**Observation:** An image $I$ can be decomposed into low-frequency (smooth) and high-frequency (detail) components:

$$I = I_{\text{smooth}} + I_{\text{detail}} \quad \text{where } I_{\text{smooth}} = I * G_\sigma$$

Therefore: $I_{\text{detail}} = I - I * G_\sigma$

### Step 2: Sharpening by Amplifying Detail

Add an amplified version of the detail back to the original:

$$I_{\text{sharp}} = I + \alpha \cdot I_{\text{detail}} = I + \alpha(I - I * G_\sigma)$$

$$= (1 + \alpha)I - \alpha(I * G_\sigma)$$

**In kernel form:**
$$K_{\text{sharp}} = (1 + \alpha)\delta - \alpha G_\sigma$$

where $\delta$ is the identity kernel (impulse).

### Step 3: Derivation of the Discrete 3×3 Sharpening Kernel

Using the $3 \times 3$ discrete Laplacian as the detail extractor:

$$\nabla^2 \approx K_L = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

The sharpened image: $I_{\text{sharp}} = I - \alpha \nabla^2 I = I * (\delta - \alpha K_L)$

$$K_{\text{sharp}} = \delta - \alpha K_L = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0 \end{bmatrix} - \alpha\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix} = \begin{bmatrix} 0 & -\alpha & 0 \\ -\alpha & 1+4\alpha & -\alpha \\ 0 & -\alpha & 0 \end{bmatrix}$$

For $\alpha = 1$: $K = \begin{bmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0 \end{bmatrix}$ — the standard sharpening kernel. $\blacksquare$

### Step 4: Proof That Sum of Kernel Elements = 1

**Theorem:** Any sharpening kernel derived from unsharp masking preserves mean brightness.

**Proof:** $\sum_{ij} K_{\text{sharp}}[i,j] = \sum_{ij} \delta[i,j] - \alpha\sum_{ij} K_L[i,j] = 1 - \alpha \cdot 0 = 1$

since $\sum K_L = 0$ (Laplacian kernel sums to zero — it's a zero-mean operator). $\blacksquare$

### Step 5: Frequency Domain Analysis of Sharpening

$$\hat{K}_{\text{sharp}}(\omega) = 1 + \alpha(1 - \hat{G}(\omega))$$

- At DC ($\omega = 0$): $\hat{K}(0) = 1$ (mean preserved)
- At high frequencies ($\omega \to \infty$): $\hat{K}(\omega) \to 1 + \alpha$ (amplified)

**Gain plot:** The sharpening kernel amplifies high frequencies by factor $1 + \alpha$ while leaving the DC component unchanged. Increasing $\alpha$ increases the sharpening strength but also amplifies noise.

### Step 6: RL Agent Parameter Selection

The sharpening strength $\alpha$ is a critical parameter:
- $\alpha = 0$: no sharpening (identity)
- $\alpha = 0.5$: mild sharpening
- $\alpha = 1.0$: moderate sharpening
- $\alpha > 2.0$: aggressive (may create halos)

An RL agent learns to set $\alpha$ based on image content and noise level, avoiding over-sharpening in noisy regions while applying strong sharpening to clean, blurry regions.

## Border Handling in Convolution — Boundary Conditions and Their Effects

When convolving near image borders, the kernel extends beyond the image. Different boundary treatments lead to different mathematical properties.

### Step 1: The Border Problem

For an $N \times N$ image and a $(2k+1) \times (2k+1)$ kernel, the output at position $(i, j)$ requires values at $(i + p, j + q)$ for $|p|, |q| \leq k$. When $i < k$ or $i \geq N - k$ (and similarly for $j$), some required positions are outside the image.

### Step 2: Common Boundary Conditions

**Zero padding:** $I_{\text{ext}}[m, n] = 0$ for $(m, n) \notin [0, M) \times [0, N)$

Equivalent to assuming the image is surrounded by black. Simple but introduces artificial edges at borders.

**Replicate (clamp):** $I_{\text{ext}}[m, n] = I[\text{clamp}(m, 0, M-1), \text{clamp}(n, 0, N-1)]$

Extends the border pixels outward. No artificial edges, but can create visible bands.

**Reflect:** $I_{\text{ext}}[m, n] = I[|m|, |n|]$ (reflecting at boundaries)

Smooth extension that preserves continuity. Most commonly used in practice.

**Wrap (periodic):** $I_{\text{ext}}[m, n] = I[m \bmod M, n \bmod N]$

Treats the image as a torus. Required for FFT-based convolution (circular convolution).

### Step 3: Proof That FFT Convolution Implements Circular Convolution

**Theorem:** $\text{IFFT}(\text{FFT}(I) \cdot \text{FFT}(K))$ computes the circular (periodic) convolution.

**Proof:** The DFT implicitly assumes periodic extension with period $N$. Therefore:
$$(I *_{\text{circ}} K)[m] = \sum_{k=0}^{N-1} I[(m-k) \bmod N] \cdot K[k]$$

This equals the linear convolution only if both $I$ and $K$ are zero-padded to length $\geq N_I + N_K - 1$ before FFT. $\blacksquare$

### Step 4: Impact on RL

For an RL agent that applies convolution-based actions:
- **Interior pixels:** Boundary condition doesn't matter (kernel fits entirely inside image)
- **Border pixels:** Different boundary conditions can significantly affect edge-region processing
- **Consistent treatment:** The agent should use the same boundary condition during training and inference

The reflect boundary condition is generally the best default for RL, as it avoids artificial edges (which could confuse the agent's state perception) while being computationally simple.

In [ ]:
def convolve2d_manual(image, kernel):
    """Manual 2D convolution - understanding the math step by step."""
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape
    
    # Padding size
    pad_h, pad_w = k_h // 2, k_w // 2
    
    # Pad the image
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
    
    # Output image
    output = np.zeros_like(image, dtype=np.float64)
    
    # Flip kernel for convolution (not cross-correlation)
    kernel_flipped = kernel[::-1, ::-1]
    
    # Slide kernel over image
    for i in range(img_h):
        for j in range(img_w):
            # Extract neighborhood
            region = padded[i:i+k_h, j:j+k_w]
            # Element-wise multiply and sum
            output[i, j] = np.sum(region * kernel_flipped)
    
    return output

# Create sample image
np.random.seed(42)
size = 64
x = np.linspace(-3, 3, size)
X, Y = np.meshgrid(x, x)
sample_img = (np.sin(X*3)**2 + np.cos(Y*2)**2)
sample_img = (sample_img / sample_img.max() * 255).astype(np.float64)

# Define various kernels
kernels = {
    'Identity': np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float64),
    'Box Blur 3×3': np.ones((3,3), dtype=np.float64) / 9,
    'Gaussian 3×3': np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float64) / 16,
    'Sharpen': np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float64),
    'Edge Detect': np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]], dtype=np.float64),
    'Emboss': np.array([[-2,-1,0],[-1,1,1],[0,1,2]], dtype=np.float64),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (name, kernel) in enumerate(kernels.items()):
    row, col = idx // 3, idx % 3
    
    # Apply convolution
    result = cv2.filter2D(sample_img, -1, kernel)
    
    axes[row, col].imshow(result, cmap='gray')
    axes[row, col].set_title(f'{name}\nKernel:\n{kernel}', fontsize=9)
    axes[row, col].axis('off')

plt.suptitle('Convolution with Different Kernels\n'
             'Each kernel = One possible RL ACTION!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('convolution_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize the convolution operation step by step
small_img = np.array([
    [10, 20, 30, 40, 50],
    [60, 70, 80, 90, 100],
    [110, 120, 130, 140, 150],
    [160, 170, 180, 190, 200],
    [210, 220, 230, 240, 250]
], dtype=np.float64)

kernel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=np.float64)  # Sobel X

print("=" * 60)
print("STEP-BY-STEP CONVOLUTION EXAMPLE")
print("=" * 60)
print(f"\nInput Image (5×5):")
print(small_img)
print(f"\nKernel (Sobel-X, 3×3):")
print(kernel)
print(f"\nComputing output at position (1,1):")
print(f"  Neighborhood: {small_img[0:3, 0:3]}")
print(f"  Flipped kernel: {kernel[::-1, ::-1]}")
print(f"  Element-wise multiply:")

region = small_img[0:3, 0:3]
k_flip = kernel[::-1, ::-1]
products = region * k_flip
print(f"  {products}")
print(f"  Sum = {products.sum():.0f}")
print(f"\n  Formula: (I*K)[1,1] = Σᵢ Σⱼ I[1+i, 1+j] · K[i, j]")
print(f"  = {' + '.join([f'{region[i,j]:.0f}×{k_flip[i,j]:.0f}' for i in range(3) for j in range(3)])}")
print(f"  = {products.sum():.0f}")

## 2. Gaussian Blur — Detailed Math

### 2D Gaussian Function:
$$G(x, y) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{x^2 + y^2}{2\sigma^2}\right)$$

### Creating the kernel:
For a $(2k+1) \times (2k+1)$ kernel with standard deviation $\sigma$:
$$K[i, j] = \frac{1}{Z} \cdot G(i-k, j-k)$$

Where $Z = \sum_{i,j} G(i-k, j-k)$ is the normalization constant.

### Separability Property:
$$G(x,y) = G(x) \cdot G(y)$$

This means 2D Gaussian can be applied as two 1D convolutions!
- Complexity: $O(n^2 \cdot k^2) \rightarrow O(n^2 \cdot 2k)$

In [ ]:
# Build Gaussian kernel from scratch
def make_gaussian_kernel(size, sigma):
    """Create 2D Gaussian kernel with full math."""
    k = size // 2
    x = np.arange(-k, k+1)
    y = np.arange(-k, k+1)
    X, Y = np.meshgrid(x, y)
    
    # 2D Gaussian
    G = (1 / (2 * np.pi * sigma**2)) * np.exp(-(X**2 + Y**2) / (2 * sigma**2))
    
    # Normalize so sum = 1
    G = G / G.sum()
    
    return G

# Show different sigma values
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sigmas = [0.5, 1.0, 2.0, 4.0]

for idx, sigma in enumerate(sigmas):
    kernel = make_gaussian_kernel(15, sigma)
    
    # Show kernel
    axes[0, idx].imshow(kernel, cmap='hot')
    axes[0, idx].set_title(f'σ = {sigma}\nKernel 15×15')
    
    # Apply to image
    blurred = cv2.filter2D(sample_img, -1, kernel)
    axes[1, idx].imshow(blurred, cmap='gray')
    axes[1, idx].set_title(f'Result (σ={sigma})')
    axes[1, idx].axis('off')

plt.suptitle('Gaussian Blur: σ controls smoothing strength\n'
             'RL Agent learns optimal σ for each image!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gaussian_blur.png', dpi=150, bbox_inches='tight')
plt.show()

## Separability of the 2D Gaussian — Proof and Computational Savings

Separability is the single most important computational optimization in image filtering. It transforms a 2D operation into two sequential 1D operations.

### Step 1: Definition of Separability

A 2D kernel $K(x, y)$ is **separable** if it can be written as:
$$K(x, y) = h(x) \cdot g(y)$$

for some 1D functions $h$ and $g$.

### Step 2: Proof That the 2D Gaussian Is Separable

$$G(x, y; \sigma) = \frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2 + y^2}{2\sigma^2}\right)$$

$$= \frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2}{2\sigma^2}\right)\exp\left(-\frac{y^2}{2\sigma^2}\right)$$

$$= \underbrace{\frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{x^2}{2\sigma^2}\right)}_{G_x(x;\sigma)} \cdot \underbrace{\frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{y^2}{2\sigma^2}\right)}_{G_y(y;\sigma)} \quad \blacksquare$$

The key algebraic fact is that $\exp(a + b) = \exp(a)\exp(b)$, and $x^2 + y^2$ separates into $x^2$ and $y^2$.

### Step 3: Separable Convolution Implementation

Instead of 2D convolution:
$$(I * K)[m, n] = \sum_{i=-k}^{k}\sum_{j=-k}^{k} I[m-i, n-j] \cdot h[i] \cdot g[j]$$

Rewrite as two passes:

**Pass 1 (horizontal):** $T[m, n] = \sum_{j=-k}^{k} I[m, n-j] \cdot g[j]$

**Pass 2 (vertical):** $O[m, n] = \sum_{i=-k}^{k} T[m-i, n] \cdot h[i]$

### Step 4: Computational Complexity Proof

For an $N \times N$ image with a $(2k+1) \times (2k+1)$ kernel:

**Non-separable:** Each output pixel requires $(2k+1)^2$ multiplications and additions.
$$T_{\text{2D}} = N^2 \cdot (2k+1)^2 \approx 4N^2k^2 \text{ operations}$$

**Separable:** Two passes, each requiring $(2k+1)$ operations per pixel:
$$T_{\text{sep}} = 2 \cdot N^2 \cdot (2k+1) \approx 4N^2k \text{ operations}$$

**Speedup factor:**
$$\frac{T_{\text{2D}}}{T_{\text{sep}}} = \frac{(2k+1)^2}{2(2k+1)} = \frac{2k+1}{2} \approx k$$

| Kernel Size | Non-Separable | Separable | Speedup |
|:------------|:-------------|:----------|:--------|
| 3×3 | 9 ops/pixel | 6 ops/pixel | 1.5× |
| 7×7 | 49 ops/pixel | 14 ops/pixel | 3.5× |
| 15×15 | 225 ops/pixel | 30 ops/pixel | 7.5× |
| 31×31 | 961 ops/pixel | 62 ops/pixel | 15.5× |

### Step 5: SVD Test for Separability

**General method:** A kernel $K$ is separable if and only if $\operatorname{rank}(K) = 1$.

**Proof:** If $K = \mathbf{h}\mathbf{g}^T$ (outer product), then $K$ has rank 1 by definition. Conversely, any rank-1 matrix can be written as an outer product. The SVD reveals this: if $K = \sigma_1\mathbf{u}_1\mathbf{v}_1^T$ with $\sigma_2 = 0$, then $K$ is separable with $h = \sqrt{\sigma_1}\mathbf{u}_1$ and $g = \sqrt{\sigma_1}\mathbf{v}_1$. $\blacksquare$

For non-separable kernels (rank > 1), we can still decompose: $K = \sum_{i=1}^r \sigma_i \mathbf{u}_i \mathbf{v}_i^T$ and apply $r$ separable convolutions — beneficial when $r \ll 2k+1$.

## Box Filter to Gaussian via the Central Limit Theorem

The Central Limit Theorem provides a beautiful connection between the simplest filter (box/averaging) and the optimal one (Gaussian).

### Step 1: Box Filter Definition

A 1D box filter of width $k$:
$$h_{\text{box}}[n] = \begin{cases} 1/k & |n| \leq (k-1)/2 \\ 0 & \text{otherwise} \end{cases}$$

This is the uniform distribution on $k$ points.

### Step 2: Repeated Box Filtering = Gaussian

**Theorem:** Convolving a box filter with itself $N$ times produces a filter that converges to a Gaussian as $N \to \infty$.

**Proof via the Central Limit Theorem:**

Each box filter represents sampling from a uniform distribution $U$. The convolution $h_{\text{box}} * h_{\text{box}} * \cdots * h_{\text{box}}$ ($N$ times) represents the distribution of the sum $S_N = U_1 + U_2 + \cdots + U_N$ of $N$ independent uniform random variables.

By the CLT: $\frac{S_N - N\mu}{\sqrt{N}\sigma} \xrightarrow{d} \mathcal{N}(0, 1)$ as $N \to \infty$.

Therefore the $N$-fold box convolution converges to a Gaussian with:
$$\mu_N = N\mu_{\text{box}}, \quad \sigma_N^2 = N\sigma_{\text{box}}^2 = N \cdot \frac{k^2 - 1}{12} \quad \blacksquare$$

### Step 3: Rate of Convergence (Berry-Esseen)

The Berry-Esseen theorem gives the convergence rate:
$$\sup_x |F_N(x) - \Phi(x)| \leq \frac{C \cdot E[|U - \mu|^3]}{\sigma^3 \sqrt{N}}$$

where $C \leq 0.4748$. For the uniform distribution: convergence is $O(1/\sqrt{N})$.

**Practical implication:** Even $N = 3$ repeated box filters gives an excellent Gaussian approximation (error < 5%).

### Step 4: Computational Advantage

**Box filter has O(1) cost per pixel** using the sliding window technique:

$$\text{sum}[n] = \text{sum}[n-1] + I[n + k/2] - I[n - k/2 - 1]$$

Therefore:
- Single box filter: $O(N^2)$ total (constant per pixel regardless of kernel size!)
- Repeated 3× box: $O(3N^2) = O(N^2)$ — approximates Gaussian at box filter cost

Compare: Direct Gaussian convolution is $O(N^2 K)$ even with separability.

### Step 5: The Binomial Filter Bridge

The discrete convolution of a box $[1, 1]$ with itself $n$ times gives binomial coefficients:
$$[1,1] * [1,1] = [1, 2, 1] \quad (\text{Sobel smoothing vector!})$$
$$[1,2,1] * [1,1] = [1, 3, 3, 1]$$
$$[1,3,3,1] * [1,1] = [1, 4, 6, 4, 1]$$

The $n$-th row of Pascal's triangle approaches a Gaussian. The Sobel smoothing weights $[1, 2, 1]$ are exactly the 2nd-order binomial filter — the simplest discrete Gaussian approximation.

This hierarchy connects: **box filter → binomial filter → Gaussian filter**, with each step closer to the theoretically optimal smoothing operation.

## Frequency Domain Interpretation — Low-Pass, High-Pass, and Band-Pass Filtering

Every spatial filter has a frequency domain interpretation. Understanding this duality reveals what each filter actually does to the image.

### Step 1: Filter as Frequency Selector

By the convolution theorem, spatial filtering $I_{\text{out}} = I * K$ is equivalent to:
$$\hat{I}_{\text{out}}(u, v) = \hat{I}(u, v) \cdot \hat{K}(u, v)$$

The filter's Fourier transform $\hat{K}(u, v)$ is called the **transfer function** or **frequency response**.

### Step 2: Low-Pass Filters (Smoothing)

A low-pass filter passes low frequencies and attenuates high frequencies:
$$|\hat{K}(u, v)| \approx \begin{cases} 1 & \text{for } \sqrt{u^2+v^2} < f_c \text{ (pass)} \\ 0 & \text{for } \sqrt{u^2+v^2} > f_c \text{ (reject)} \end{cases}$$

**Gaussian blur transfer function:**
$$\hat{G}(u, v; \sigma) = \exp\left(-\frac{(u^2 + v^2)\sigma^2}{2}\right)$$

*Proof:* The Fourier transform of a Gaussian is a Gaussian. Larger $\sigma$ in space → narrower passband → more smoothing. $\blacksquare$

**Box filter transfer function:**
$$\hat{B}(u, v) = \text{sinc}(u \cdot w/2) \cdot \text{sinc}(v \cdot w/2)$$

The sinc function has sidelobes → box filter has ringing artifacts (Gibbs phenomenon).

### Step 3: High-Pass Filters (Sharpening / Edge Detection)

A high-pass filter suppresses low frequencies and passes high frequencies.

**The unsharp mask** in frequency domain:
$$\hat{K}_{\text{sharp}} = 1 + \alpha(1 - \hat{G}) = (1 + \alpha) - \alpha\hat{G}$$

At DC ($u = v = 0$): $\hat{K}(0,0) = 1$ (preserves mean brightness)
At high frequencies: $\hat{K} \approx 1 + \alpha$ (amplifies by factor $1 + \alpha$)

**The Laplacian** in frequency domain:
$$\widehat{\nabla^2 I}(u,v) = -(u^2 + v^2)\hat{I}(u,v)$$

The transfer function $H(u,v) = -(u^2 + v^2)$ increases quadratically with frequency — it's a high-pass filter that responds more strongly to higher frequencies.

### Step 4: Band-Pass Filters

A band-pass filter passes frequencies in a specific range $[f_1, f_2]$. The Difference of Gaussians (DoG) is a natural band-pass:

$$\hat{D}(u,v) = \hat{G}(u,v;\sigma_1) - \hat{G}(u,v;\sigma_2)$$

$$= \exp(-\sigma_1^2(u^2+v^2)/2) - \exp(-\sigma_2^2(u^2+v^2)/2)$$

For $\sigma_1 < \sigma_2$: this passes frequencies between $\sim 1/\sigma_2$ and $\sim 1/\sigma_1$.

### Step 5: Ideal vs. Practical Filters

**Ideal low-pass** (perfect cutoff): $H(f) = \mathbf{1}_{f \leq f_c}$

Its inverse FT is $\text{sinc}(2\pi f_c r)$ — an infinitely wide, oscillating kernel. This is:
1. Non-causal (requires future samples)
2. Causes ringing (Gibbs phenomenon)
3. Cannot be implemented exactly

**Gaussian** avoids these problems: smooth frequency response → smooth spatial kernel → no ringing. This is another reason Gaussian is the preferred low-pass filter.

### Step 6: RL Kernel Design in Frequency Domain

An RL agent designing custom filters can parameterize the transfer function:
$$\hat{K}(u, v; \boldsymbol{\theta}) = \sum_{i=1}^{M} w_i \exp\left(-\frac{(u^2+v^2)}{2\sigma_i^2}\right)$$

The agent learns weights $w_i$ and bandwidths $\sigma_i$ to create task-specific frequency responses — automatically discovering the optimal filtering strategy for each image processing task.

## 3. RL Formulation: Filter Selection Agent

### MDP for Filter Application:
- **State:** Current image $I_t$ (or its histogram/features)
- **Actions:** $\{\text{blur}_{\sigma=1}, \text{blur}_{\sigma=2}, \text{sharpen}, \text{edge}, \text{identity}, ...\}$
- **Transition:** $I_{t+1} = K_{a_t} * I_t$ (apply selected kernel)
- **Reward:** Quality improvement $r_t = Q(I_{t+1}) - Q(I_t)$

The agent learns **which filter to apply and when** — no hand-crafted rules needed!

---
**Next:** Module 2.2 — Edge Detection